# 03 2.5D Qwen2.5-VL SFT Train

Inputs:

- output dataset from notebook `01_build_25d_cache.ipynb`
- Qwen2.5-VL model available through Kaggle model/input or Hugging Face

This version is configured for a stricter Q1 run on Kaggle T4 x2. It uses QLoRA 4-bit,
gradient checkpointing, answer-only label masking, duplicate-target capping, and real
metadata context from the manifest. The duplicate cap is necessary because one generic
Findings report appears hundreds of times in the training split and otherwise causes the
model to collapse into a single generic report.

The fixed manifest parses both Findings and Impression when present. The training target uses
structured `FINDINGS:` and `IMPRESSION:` headings and does not synthesize missing sections.


In [ ]:
# Environment setup for Kaggle.
# Keep Internet ON if the model/packages are not already attached as Kaggle inputs.

from pathlib import Path
import json, warnings, os, sys, subprocess, importlib, re, gc, math

os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
warnings.filterwarnings("ignore")

MIN_PACKAGES = {
    "transformers": "4.51.0",
    "accelerate": "0.34.0",
    "peft": "0.12.0",
    "bitsandbytes": "0.46.1",
    "qwen-vl-utils": "0.0.8",
    "pillow": "10.0.0",
}

def version_tuple(v):
    nums = re.findall(r"\d+", str(v))[:4]
    return tuple(int(x) for x in nums) if nums else (0,)

def installed_version(package_name):
    try:
        from importlib import metadata
        return metadata.version(package_name)
    except Exception:
        return None

def ensure_package(package_name, min_version):
    current = installed_version(package_name)
    if current is not None and version_tuple(current) >= version_tuple(min_version):
        print(f"{package_name}: {current} OK")
        return
    spec = f"{package_name}>={min_version}"
    print(f"Installing {spec} (current={current})")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", spec])

for pkg, min_ver in MIN_PACKAGES.items():
    ensure_package(pkg, min_ver)

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from IPython.display import display

# Use RUN_MODE="smoke" only for quick environment checks. Full is the Q1 training run.
RUN_MODE = os.environ.get("RUN_MODE", "full").lower()
SMOKE_MODE = RUN_MODE == "smoke"
MODEL_ID_OR_PATH = os.environ.get("QWEN_MODEL_PATH", "Qwen/Qwen2.5-VL-3B-Instruct")
USE_4BIT_QLORA = True
ALLOW_FP16_FALLBACK = False

DUPLICATE_TARGET_CAP = 2 if SMOKE_MODE else 8
MAX_TRAIN_SAMPLES = 128 if SMOKE_MODE else None
MAX_VAL_SAMPLES = 64 if SMOKE_MODE else 160
TRAIN_EPOCHS = 1 if SMOKE_MODE else 3
MAX_STEPS = 40 if SMOKE_MODE else -1
EVAL_SAVE_STEPS = 20 if SMOKE_MODE else 100
LEARNING_RATE = 2e-4 if SMOKE_MODE else 7e-5
GEN_EVAL_SAMPLES = 8 if SMOKE_MODE else 48

OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
SFT_DIR = OUTPUT_ROOT / "03_qwen_sft"
SFT_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_ID_OR_PATH:", MODEL_ID_OR_PATH)
print("RUN_MODE:", RUN_MODE, "USE_4BIT_QLORA:", USE_4BIT_QLORA)
print("DUPLICATE_TARGET_CAP:", DUPLICATE_TARGET_CAP, "TRAIN_EPOCHS:", TRAIN_EPOCHS, "LR:", LEARNING_RATE)
print("CUDA available:", torch.cuda.is_available(), "GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(i, torch.cuda.get_device_name(i), round(props.total_memory / 1024**3, 2), "GiB")


In [ ]:
def locate_cache():
    roots = list(Path("/kaggle/input").rglob("q1_cache_meta.json")) if Path("/kaggle/input").exists() else []
    roots += list(Path.cwd().rglob("q1_cache_meta.json"))
    if not roots:
        raise FileNotFoundError("q1_cache_meta.json not found. Add notebook 01 output.")
    meta_path = sorted(roots, key=lambda p: len(str(p)))[0]
    return meta_path.parent, json.loads(meta_path.read_text())

CACHE_DIR, meta = locate_cache()
clean = pd.read_csv(CACHE_DIR / "q1_clean_manifest_for_cache.csv")
cache_index = pd.read_csv(CACHE_DIR / "q1_cache_index.csv")
clean = clean.drop(columns=["row_index"], errors="ignore").merge(
    cache_index[cache_index.cache_ok][["case_id", "row_index"]], on="case_id", how="inner"
)

target_col = meta.get("target_column", "target_text")
if target_col in clean.columns:
    clean["base_report_text"] = clean[target_col].fillna("").astype(str)
elif "report_text_clean" in clean.columns:
    clean["base_report_text"] = clean["report_text_clean"].fillna("").astype(str)
else:
    clean["base_report_text"] = clean["target_text"].fillna("").astype(str)

def clean_value(x):
    s = str(x or "").strip()
    if s.lower() in {"nan", "none", "null"}:
        return ""
    return re.sub(r"\s+", " ", s)

def build_context(row):
    items = []
    for label, col in [("Sex", "sex"), ("Birth year", "birth_year"), ("Indication", "indication"), ("Clinical history", "clinical_history")]:
        val = clean_value(row.get(col, ""))
        if val:
            items.append(f"{label}: {val}")
    return "\n".join(items) if items else "No structured clinical context available."

def build_sft_target(row):
    findings = clean_value(row.get("findings", ""))
    impression = clean_value(row.get("impression", ""))
    parts = []
    if findings:
        parts.append("FINDINGS:\n" + findings)
    if impression:
        parts.append("IMPRESSION:\n" + impression)
    target = "\n\n".join(parts).strip()
    if not target:
        target = clean_value(row.get("base_report_text", ""))
    return target

clean["case_context"] = clean.apply(build_context, axis=1)
clean["target_text"] = clean.apply(build_sft_target, axis=1)
arr = np.memmap(
    CACHE_DIR / meta["mmap_path"],
    dtype=np.uint8,
    mode="r",
    shape=(meta["n"], meta["channels"], meta["image_size"], meta["image_size"]),
)

section_check = {
    "rows": int(len(clean)),
    "findings_present": int(clean["findings"].fillna("").astype(str).str.strip().ne("").sum()),
    "impression_present": int(clean["impression"].fillna("").astype(str).str.strip().ne("").sum()),
    "target_has_impression": int(clean["target_text"].str.contains("IMPRESSION:", regex=False).sum()),
}
section_check["impression_coverage"] = section_check["impression_present"] / max(section_check["rows"], 1)
print("section_check", json.dumps(section_check, ensure_ascii=False, indent=2))
assert section_check["impression_coverage"] >= 0.90, section_check

train_all = clean[clean.clean_split == "train"].copy()
val_df = clean[clean.clean_split == "validation"].copy()
# Cap duplicate targets to avoid one frequent report dominating generation.
train_df = (
    train_all.groupby("target_text", group_keys=False, dropna=False)
    .apply(lambda x: x.sample(min(len(x), DUPLICATE_TARGET_CAP), random_state=391))
    .reset_index(drop=True)
)
if MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=391)
if MAX_VAL_SAMPLES:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=392)

train_stats = {
    "train_all_rows": int(len(train_all)),
    "train_rows_after_duplicate_cap": int(len(train_df)),
    "train_unique_targets": int(train_df["target_text"].nunique()),
    "train_top_target_count_after_cap": int(train_df["target_text"].value_counts().iloc[0]),
    "validation_rows_used_for_loss": int(len(val_df)),
    **section_check,
}
print(json.dumps(train_stats, ensure_ascii=False, indent=2))
print("cache", CACHE_DIR)
print("target preview:")
print(train_df["target_text"].iloc[0][:800])
print("context preview:")
print(train_df["case_context"].iloc[0][:500])

def row_to_rgb(row_index):
    x = arr[int(row_index)]
    # RGB mosaic: CT middle, PET coronal, PET axial.
    rgb = np.stack([x[1], x[4], x[3]], axis=-1)
    return Image.fromarray(rgb, mode="RGB")


In [ ]:
from transformers import AutoProcessor, TrainingArguments, Trainer, BitsAndBytesConfig
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as QwenModel
except Exception:
    from transformers import AutoModelForVision2Seq as QwenModel
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 03 requires GPU. In Kaggle Settings, set Accelerator = GPU T4 x2.")

processor = AutoProcessor.from_pretrained(MODEL_ID_OR_PATH, trust_remote_code=True, use_fast=False)
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "right"

quantization_config = None
if USE_4BIT_QLORA:
    try:
        import bitsandbytes as bnb
        print("bitsandbytes:", getattr(bnb, "__version__", "unknown"))
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    except Exception as exc:
        if not ALLOW_FP16_FALLBACK:
            raise RuntimeError(
                "bitsandbytes is required for QLoRA 4-bit on T4. Keep Internet ON and run the setup cell."
            ) from exc
        print("bitsandbytes unavailable; falling back to fp16. This may OOM on T4.", repr(exc))

max_memory = {i: "14GiB" for i in range(torch.cuda.device_count())}
max_memory["cpu"] = "32GiB"
if quantization_config is not None:
    model = QwenModel.from_pretrained(
        MODEL_ID_OR_PATH,
        quantization_config=quantization_config,
        device_map="auto",
        max_memory=max_memory,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = QwenModel.from_pretrained(
        MODEL_ID_OR_PATH,
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=max_memory,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False

try:
    model.gradient_checkpointing_enable()
except Exception as exc:
    print("gradient_checkpointing_enable warning:", repr(exc))

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print("device map:", getattr(model, "hf_device_map", None))


In [ ]:
PROMPT_PREFIX = (
    "Generate a structured Vietnamese PET/CT report from the image and clinical context. "
    "Use exactly two headings when available: FINDINGS and IMPRESSION. "
    "Preserve clinically relevant organs, lesions, and SUV values when visible in the provided reference distribution."
)

def build_user_prompt(row):
    return PROMPT_PREFIX + "\n\nClinical context:\n" + str(row.case_context)

class PetReportDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = row_to_rgb(row.row_index)
        target = str(row.target_text).strip()
        prompt = build_user_prompt(row)
        user_messages = [
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}
        ]
        full_messages = user_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": target}]}
        ]
        prompt_text = processor.apply_chat_template(user_messages, tokenize=False, add_generation_prompt=True)
        full_text = processor.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
        return {"prompt_text": prompt_text, "full_text": full_text, "image": image, "target": target, "case_id": row.case_id}

def collate(batch):
    full_texts = [b["full_text"] for b in batch]
    images = [b["image"] for b in batch]
    enc = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100

    # Mask user/image prompt tokens. The loss is computed only on the assistant report text.
    for i, b in enumerate(batch):
        prompt_enc = processor(text=[b["prompt_text"]], images=[b["image"]], return_tensors="pt")
        prompt_len = int(prompt_enc["input_ids"].shape[1])
        labels[i, :min(prompt_len, labels.shape[1])] = -100
    enc["labels"] = labels
    return enc

args = TrainingArguments(
    output_dir=str(SFT_DIR / "checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    num_train_epochs=TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    logging_steps=10 if not SMOKE_MODE else 5,
    save_steps=EVAL_SAVE_STEPS,
    eval_steps=EVAL_SAVE_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=PetReportDataset(train_df),
    eval_dataset=PetReportDataset(val_df),
    data_collator=collate,
)

train_result = trainer.train()
trainer.save_model(SFT_DIR / "adapter_final")
processor.save_pretrained(SFT_DIR / "processor")

log_df = pd.DataFrame(trainer.state.log_history)
log_df.to_csv(SFT_DIR / "trainer_log_history.csv", index=False, encoding="utf-8-sig")
summary = {
    "run_mode": RUN_MODE,
    "model_id_or_path": MODEL_ID_OR_PATH,
    "duplicate_target_cap": int(DUPLICATE_TARGET_CAP),
    **train_stats,
    "global_step": int(trainer.state.global_step),
    "train_metrics": train_result.metrics,
    "last_log": trainer.state.log_history[-1] if trainer.state.log_history else {},
}
(SFT_DIR / "training_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Saved:", SFT_DIR)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Small qualitative validation generation. This is not the final evaluation;
# notebook 04 performs the structured validation/test evaluation after the full run.

def first_model_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def generate_report(row, max_new_tokens=768):
    image = row_to_rgb(row.row_index)
    prompt = build_user_prompt(row)
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")
    device = first_model_device(model)
    inputs = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            no_repeat_ngram_size=5,
        )
    gen = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return gen.strip()

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x or "").strip())

def lcs_len(a, b):
    dp = [0] * (len(b) + 1)
    for ca in a:
        prev = 0
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev + 1 if ca == cb else max(dp[j], dp[j - 1])
            prev = cur
    return dp[-1]

def rouge_l_f1(pred, ref):
    pred_toks = normalize_text(pred).split()
    ref_toks = normalize_text(ref).split()
    if not pred_toks or not ref_toks:
        return 0.0
    lcs = lcs_len(pred_toks, ref_toks)
    p = lcs / len(pred_toks)
    r = lcs / len(ref_toks)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

sample_df = clean[clean.clean_split == "validation"].sample(min(GEN_EVAL_SAMPLES, (clean.clean_split == "validation").sum()), random_state=493).copy()
rows = []
for _, row in sample_df.iterrows():
    pred = generate_report(row)
    ref = str(row.target_text)
    rows.append({
        "case_id": row.case_id,
        "prediction": pred,
        "target_text": ref,
        "rouge_l": rouge_l_f1(pred, ref),
        "has_findings_heading": int("findings" in pred.lower()),
        "has_impression_heading": int("impression" in pred.lower()),
        "prediction_word_count": len(normalize_text(pred).split()),
        "target_word_count": len(normalize_text(ref).split()),
    })

gen_df = pd.DataFrame(rows)
gen_df.to_csv(SFT_DIR / "validation_generation_samples.csv", index=False, encoding="utf-8-sig")
if len(gen_df):
    unique_ratio = float(gen_df["prediction"].nunique() / len(gen_df))
    top_fraction = float(gen_df["prediction"].value_counts().iloc[0] / len(gen_df))
    gen_summary = {
        **gen_df[["rouge_l", "has_findings_heading", "has_impression_heading", "prediction_word_count", "target_word_count"]].mean(numeric_only=True).to_dict(),
        "unique_prediction_ratio": unique_ratio,
        "top_prediction_fraction": top_fraction,
        "exact_match_count": int((gen_df["prediction"].str.strip() == gen_df["target_text"].str.strip()).sum()),
        "sample_count": int(len(gen_df)),
        "collapse_warning": bool(unique_ratio < 0.50 or top_fraction > 0.50),
    }
else:
    gen_summary = {}
(SFT_DIR / "validation_generation_summary.json").write_text(json.dumps(gen_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(gen_summary, ensure_ascii=False, indent=2))
display(gen_df[["case_id", "rouge_l", "has_findings_heading", "has_impression_heading", "prediction_word_count", "target_word_count", "prediction", "target_text"]].head())
